In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import cleaning  # noqa: E402
from src.utils import (  # noqa: E402
    summarize_dataframe_health,
    train_val_test_split,
)

RAW = ROOT / "data" / "raw" / "dataset.csv"
CLEAN = ROOT / "data" / "clean" / "clean.parquet"

# 01 Cleaning: Attenomy telemetry dataset
Target: `is_anomaly`. Deterministic fixes run in `src/cleaning.py`. Anything that learns a statistic (median, scaler, encoder) is fit on the **training split only**.

## 1. Profile the raw data

In [ ]:
raw = pd.read_csv(RAW)
h = summarize_dataframe_health(raw)
print(
    f"Rows: {h['n_rows']:,} | Columns: {h['n_cols']} | "
    f"Memory: {h['memory_mb']} MB | Duplicate rows: {h['duplicate_rows']}"
)
pd.DataFrame({"dtype": h["dtypes"], "null_pct": h["null_pct"]})

In [ ]:
print("Raw channel values:", sorted(raw["channel"].dropna().unique()))
print("Negative session durations:", int((raw["session_duration_sec"] < 0).sum()))
print("Duplicate rows:", int(raw.duplicated().sum()))
print("Duplicate transaction_ids:", int(raw["transaction_id"].duplicated().sum()))
bad_keys = ~raw["transaction_id"].str.match(cleaning.KEY_PATTERN)
print("Malformed transaction_ids:", int(bad_keys.sum()))
print("error_code null %:", round(raw["error_code"].isna().mean() * 100, 1))
print("Target balance:", raw["is_anomaly"].value_counts(normalize=True).round(3).to_dict())

## 2. Clean (deterministic steps only)
Runs the same functions as `python src/cleaning.py`, so the notebook and the script cannot drift apart. Missing `error_code` means no error occurred (`NO_ERROR`). Negative durations become NaN. Outliers are flagged, not removed.

In [ ]:
clean_df = cleaning.clean(str(RAW), str(CLEAN))

In [ ]:
clean_df = pd.read_parquet(CLEAN)
hc = summarize_dataframe_health(clean_df)
print(f"Rows: {hc['n_rows']:,} | Memory: {hc['memory_mb']} MB")
print(f"Parquet size on disk: {CLEAN.stat().st_size / 1024:.0f} KB")
nulls = clean_df.isna().sum()
print("Columns with nulls:", nulls[nulls > 0].to_dict())
assert set(nulls[nulls > 0].index) == {"session_duration_sec"}
clean_df.dtypes

## 3. Leak-free preprocessing
Split **first**, then fit the imputer, scaler and encoders on the training rows only. The only remaining nulls are in `session_duration_sec` (explained: raw nulls plus invalid negatives), so they are filled with the train median.

In [ ]:
train, val, test = train_val_test_split(
    clean_df, (0.7, 0.15, 0.15), seed=42, stratify_col="is_anomaly"
)
print("Sizes:", len(train), len(val), len(test), "| total:", len(clean_df))
print("Anomaly rate:", [round(d["is_anomaly"].mean(), 4) for d in (train, val, test)])
assert set(train.index).isdisjoint(val.index)
assert set(train.index).isdisjoint(test.index)
assert set(val.index).isdisjoint(test.index)
print("Index sets are disjoint.")

In [ ]:
NUM_COLS = ["session_duration_sec", "payload_size_kb", "latency_ms", "retry_count"]
NOMINAL = ["interaction_type", "channel", "error_code", "region"]
ORDINAL = ["client_tier"]
TIER_ORDER = [["UNKNOWN", "starter", "pro", "enterprise", "vip"]]
FEATURES = NUM_COLS + NOMINAL + ORDINAL

numeric_pipe = Pipeline(
    [("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]
)
preprocess = ColumnTransformer(
    [
        ("num", numeric_pipe, NUM_COLS),
        ("nominal", OneHotEncoder(handle_unknown="ignore", sparse_output=False), NOMINAL),
        (
            "ordinal",
            OrdinalEncoder(
                categories=TIER_ORDER,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ORDINAL,
        ),
    ]
)


def as_text(df):
    """Cast categorical columns to plain strings for sklearn."""
    return df[FEATURES].astype(dict.fromkeys(NOMINAL + ORDINAL, str))


preprocess.fit(as_text(train))
X_train = preprocess.transform(as_text(train))
X_val = preprocess.transform(as_text(val))
X_test = preprocess.transform(as_text(test))

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("NaNs after preprocessing:", [int(np.isnan(x).sum()) for x in (X_train, X_val, X_test)])

In [ ]:
imputer = preprocess.named_transformers_["num"].named_steps["impute"]
print("Median used for session_duration_sec (train only):", imputer.statistics_[0])
print("Median over all rows (NOT used):", clean_df["session_duration_sec"].median())
print("Feature count:", len(preprocess.get_feature_names_out()))

## Findings
- No corrupted keys, duplicates or malformed timestamps were found. The checks are in `src/cleaning.py`.
- Median imputation is used because session duration is right-skewed, and the median resists extreme values.
- The imputer, scaler and encoders are fit on train rows only. Validation and test rows are only transformed.
- Outlier flags are for auditing and EDA. They are not model features, because their IQR fences used all rows.